# AllSortsHub Cartoon Studio — Episode 1 YouTube Generator

**T4-safe production mode:** Stable Video Diffusion XT 1.1 image-to-video, with chained continuation segments, 1080p assembly, audio, captions and Shorts.


In [ ]:
# 1. GPU check
!nvidia-smi
import torch, shutil
if not torch.cuda.is_available(): raise RuntimeError('No CUDA GPU. Choose Runtime > Change runtime type > GPU.')
GPU_NAME=torch.cuda.get_device_name(0); VRAM_GB=torch.cuda.get_device_properties(0).total_memory/1024**3
print('PyTorch:',torch.__version__); print('GPU:',GPU_NAME); print('VRAM:',round(VRAM_GB,1),'GB'); print('FFmpeg:',shutil.which('ffmpeg'))
if VRAM_GB < 8: raise RuntimeError('At least 8 GB VRAM is required.')
print('Generation path: Stable Video Diffusion XT 1.1 production mode')

In [ ]:
# 2. Persistent Drive + project
from google.colab import drive
drive.mount('/content/drive')
BASE='/content/drive/MyDrive/AllSortsHub-Wan2.2'
!mkdir -p "$BASE/models" "$BASE/generated" "$BASE/output"
%cd /content
!rm -rf cartoon-studio
!git clone -q https://github.com/parth01/AllSortsHub-Cartoon-Studio.git cartoon-studio
!python -m pip install -q -U 'diffusers>=0.30.0,<0.36.0' transformers accelerate safetensors huggingface_hub
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
# 3. Production SVD settings
MODEL_ID='stabilityai/stable-video-diffusion-img2vid-xt-1-1'
VID_WIDTH,VID_HEIGHT=576,320
VID_FRAMES=25
VID_FPS=5
VID_STEPS=22
DECODE_CHUNK=4
print(f'SVD-XT 1.1: {VID_WIDTH}x{VID_HEIGHT}, {VID_FRAMES} frames @ {VID_FPS} fps = 5.0s segments, {VID_STEPS} steps')

In [ ]:
# 4. Authenticate to Hugging Face and load SVD-XT
import torch, gc
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN=None
if not HF_TOKEN: raise RuntimeError('HF_TOKEN is missing. Accept SVD-XT 1.1 terms on Hugging Face, then add HF_TOKEN to Colab Secrets.')
print('Loading Stable Video Diffusion XT 1.1...')
pipe=StableVideoDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, variant='fp16', token=HF_TOKEN)
pipe.enable_model_cpu_offload()
if hasattr(pipe,'vae') and hasattr(pipe.vae,'enable_slicing'): pipe.vae.enable_slicing()
if hasattr(pipe,'vae') and hasattr(pipe.vae,'enable_tiling'): pipe.vae.enable_tiling()
print('SVD-XT pipeline ready.')

In [ ]:
# 5. Prepare Episode 1 and restore production checkpoints
from pathlib import Path
import json, shutil, re
ROOT=Path('/content/cartoon-studio/master-version/AllSortsHub-Billion 2'); LOCAL_GEN=ROOT/'wan_i2v'/'generated'; DRIVE_GEN=Path(BASE)/'generated'
LOCAL_GEN.mkdir(parents=True,exist_ok=True); DRIVE_GEN.mkdir(parents=True,exist_ok=True)
manifest=json.loads((ROOT/'wan_i2v'/'manifest.json').read_text())
for p in DRIVE_GEN.glob('shot_*.mp4'):
    t=LOCAL_GEN/p.name
    if not t.exists() or t.stat().st_size<10000: shutil.copy2(p,t)
for p in DRIVE_GEN.glob('shot_*[ab].mp4'):
    t=LOCAL_GEN/p.name
    if not t.exists() or t.stat().st_size<10000: shutil.copy2(p,t)
print('Episode shots:',len(manifest['shots']))
print('Existing single clips:',len(list(LOCAL_GEN.glob('shot_??.mp4'))))
print('Existing segment clips:',len(list(LOCAL_GEN.glob('shot_??[ab].mp4'))))

In [ ]:
# 6. Production generation: 5-second clips + chained continuations
import gc, shutil, subprocess, re, torch
from PIL import Image
from diffusers.utils import export_to_video
prompt_text=(ROOT/'wan_i2v'/'prompts.txt').read_text()
STYLE='Modern 2D cel-shaded cartoon animation, bold clean black linework, semi-flat shading, vibrant colors, expressive anime-influenced facial acting, preserve the exact character designs and environment in the input image. Natural hand-drawn animation feel. Keep faces, hair, clothing, proportions, props and background layout consistent. Smooth readable physical motion, stable camera, cinematic cartoon timing.'
NEG='No photorealism, no 3D CGI, no live action, no extra fingers, no duplicate limbs, no warped faces, no character morphing, no costume changes, no hairstyle changes, no background replacement, no random objects, no random text, no logos, no watermark, no scene cuts, no sudden camera spins, no extreme deformation.'
def shot_prompt(n):
    pat=rf'SHOT {n:02d} — .*?(?=\n\nSHOT |\n\nNEGATIVE|$)'
    m=re.search(pat,prompt_text,re.S)
    return (m.group(0) if m else f'SHOT {n:02d}') + '\n' + STYLE + '\n' + NEG
def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.ipc_collect()
def generate_segment(image,out,seed):
    if out.exists() and out.stat().st_size>10000:
        print('SKIP',out.name); return
    print('GENERATING',out.name,'from',image.name)
    cleanup_gpu()
    img=Image.open(image).convert('RGB').resize((VID_WIDTH,VID_HEIGHT))
    gen=torch.Generator(device='cuda').manual_seed(seed)
    result=pipe(image=img,height=VID_HEIGHT,width=VID_WIDTH,num_frames=VID_FRAMES,num_inference_steps=VID_STEPS,min_guidance_scale=1.0,max_guidance_scale=2.6,motion_bucket_id=127,noise_aug_strength=0.015,generator=gen,decode_chunk_size=DECODE_CHUNK)
    export_to_video(result.frames[0],str(out),fps=VID_FPS)
    cleanup_gpu()
def last_frame(video,out_image):
    video=Path(video); out_image=Path(out_image); out_image.parent.mkdir(parents=True,exist_ok=True)
    # Avoid -sseof: the Colab FFmpeg build can report a valid MP4 but return zero frames when seeking from EOF.
    # The generated clips are exactly 25 frames at 5 fps, so select the final decoded frame directly.
    cmd=['ffmpeg','-y','-i',str(video),'-vf','select=eq(n\,24)','-vsync','vfr','-frames:v','1','-q:v','2',str(out_image)]
    r=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
    if r.returncode!=0 or not out_image.exists() or out_image.stat().st_size<1000:
        print('FFmpeg last-frame extraction failed:')
        print(r.stderr[-3000:])
        raise RuntimeError(f'Could not extract continuation frame from {video}')
    print('Continuation frame created:',out_image)
    return out_image
for shot in manifest['shots']:
    n=int(shot['id']); target=float(shot.get('duration',5)); image=ROOT/shot['image']
    a=LOCAL_GEN/f'shot_{n:02d}a.mp4'
    generate_segment(image,a,910000+n*10)
    if a.exists(): shutil.copy2(a,DRIVE_GEN/a.name)
    if target>5.0:
        cont=LOCAL_GEN/f'_continuation_{n:02d}.jpg'
        last_frame(a,cont)
        b=LOCAL_GEN/f'shot_{n:02d}b.mp4'
        generate_segment(cont,b,920000+n*10)
        if b.exists(): shutil.copy2(b,DRIVE_GEN/b.name)
    print(f'SHOT {n:02d} ready: {target:.1f}s target')
    cleanup_gpu()
print('Production generation complete. Run Cell 7.')

In [ ]:
# 7. Assemble final Episode 1
%cd /content/cartoon-studio/master-version/AllSortsHub-Billion 2
!python3 wan_i2v/assemble_episode.py
!cp -f output/AllSortsHub_Episode_01_WAN_MASTER.mp4 "$BASE/output/"
!cp -f output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4 "$BASE/output/"
!ls -lh output/AllSortsHub_Episode_01_WAN_MASTER.mp4 output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4

## Resume after disconnect
Rerun Cells 1–5, then Cell 6. Existing `shot_XXa.mp4` and `shot_XXb.mp4` files in Drive are restored and skipped.
